In [7]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

In [8]:
base = Path("../data")

countries = pd.read_csv(base / "CountriesSD.csv")
summer    = pd.read_csv(base / "SummerSD.csv")
winter    = pd.read_csv(base / "WinterSD.csv")
athletes  = pd.read_csv(base / "olympic_athletes.csv")
gdp = pd.read_csv(base / "gdp_per_capita.csv", skiprows=4, encoding="utf-8-sig")

In [9]:
import pandas as pd
import unicodedata

# ----------------------------
# Helper functions
# ----------------------------
def clean_columns(df):
    """Standardize column names."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return df


def clean_athlete_name(series):
    """Normalize athlete names to improve matching across datasets."""
    return (
        series.astype(str)
        .str.strip()
        .map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8"))
        .str.lower()
        .str.replace("-", " ", regex=False)
        .str.replace(r"[.,]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


# ----------------------------
# Clean tables
# ----------------------------
countries = clean_columns(countries)
summer = clean_columns(summer)
winter = clean_columns(winter)
athletes = clean_columns(athletes)
gdp = clean_columns(gdp)

for df in [countries, summer, winter, athletes, gdp]:
    df.drop(
        columns=[col for col in df.columns if col.startswith("unnamed")],
        inplace=True,
        errors="ignore"
    )


# ----------------------------
# Harmonize column names
# ----------------------------
winter = winter.rename(columns={"country": "code"})
athletes = athletes.rename(columns={"athlete_full_name": "athlete"})

summer["season"] = "Summer"
winter["season"] = "Winter"


# ----------------------------
# Clean merge keys
# ----------------------------
summer["code"] = summer["code"].astype(str).str.strip().str.upper()
winter["code"] = winter["code"].astype(str).str.strip().str.upper()
countries["code"] = countries["code"].astype(str).str.strip().str.upper()

summer["year"] = pd.to_numeric(summer["year"], errors="coerce")
winter["year"] = pd.to_numeric(winter["year"], errors="coerce")

summer["athlete_clean"] = clean_athlete_name(summer["athlete"])
winter["athlete_clean"] = clean_athlete_name(winter["athlete"])
athletes["athlete_clean"] = clean_athlete_name(athletes["athlete"])


# ----------------------------
# Reshape GDP data to long format
# ----------------------------
# Remove an existing gdp_per_capita column if present,
# because it conflicts with melt(value_name=...)
gdp = gdp.drop(columns=["gdp_per_capita"], errors="ignore")

gdp = gdp.melt(
    id_vars=["country_name", "country_code", "indicator_name", "indicator_code"],
    var_name="year",
    value_name="gdp_pc"
)

gdp = gdp[gdp["year"].astype(str).str.fullmatch(r"\d{4}")].copy()
gdp["year"] = pd.to_numeric(gdp["year"], errors="coerce")
gdp["gdp_pc"] = pd.to_numeric(gdp["gdp_pc"], errors="coerce")

gdp["code"] = gdp["country_code"].astype(str).str.strip().str.upper()

gdp = (
    gdp[["code", "year", "gdp_pc"]]
    .drop_duplicates(subset=["code", "year"])
    .rename(columns={"gdp_pc": "gdp_per_capita"})
)


# ----------------------------
# Combine Summer + Winter
# ----------------------------
olympics = pd.concat([summer, winter], ignore_index=True)


# ----------------------------
# Merge country info
# ----------------------------
country_info = (
    countries[["country", "code", "population"]]
    .drop_duplicates(subset=["code"])
)

olympics = olympics.merge(
    country_info,
    on="code",
    how="left",
    suffixes=("", "_country")
)

if "country_country" in olympics.columns:
    olympics["country"] = olympics["country"].fillna(olympics["country_country"])
    olympics = olympics.drop(columns=["country_country"])


# ----------------------------
# Merge athlete birth year
# ----------------------------
athlete_info = (
    athletes[["athlete_clean", "athlete_year_birth"]]
    .dropna(subset=["athlete_clean"])
    .drop_duplicates(subset=["athlete_clean"])
)

olympics = olympics.merge(
    athlete_info,
    on="athlete_clean",
    how="left"
)

olympics["athlete_year_birth"] = pd.to_numeric(
    olympics["athlete_year_birth"],
    errors="coerce"
)
olympics["age"] = olympics["year"] - olympics["athlete_year_birth"]


# ----------------------------
# Merge GDP by Olympic year
# ----------------------------
olympics = olympics.merge(
    gdp,
    on=["code", "year"],
    how="left"
)


# ----------------------------
# Final cleanup
# ----------------------------
olympics = olympics.drop(columns=["athlete_clean"], errors="ignore")

olympics.to_csv("../data/olympics.csv", index=False)

display(olympics.head())

,year,city,sport,discipline,athlete,code,gender,event,medal,country,season,population,athlete_year_birth,age,gdp_per_capita
0,1896,Athens,Aquatics,Swimming,Alfred Hajos,HUN,Men,100M Freestyle,Gold,Hungary,Summer,9844686.0,1878.0,18.0,NaN
1,1896,Athens,Aquatics,Swimming,Otto Herschmann,AUT,Men,100M Freestyle,Silver,Austria,Summer,8611088.0,1877.0,19.0,NaN
2,1896,Athens,Aquatics,Swimming,Dimitrios Drivas,GRE,Men,100M Freestyle For Sailors,Bronze,Greece,Summer,10823732.0,NaN,NaN,NaN
3,1896,Athens,Aquatics,Swimming,Ioannis Malokinis,GRE,Men,100M Freestyle For Sailors,Gold,Greece,Summer,10823732.0,NaN,NaN,NaN
4,1896,Athens,Aquatics,Swimming,Spiridon Chasapis,GRE,Men,100M Freestyle For Sailors,Silver,Greece,Summer,10823732.0,NaN,NaN,NaN
